In [ ]:
pip install sqlalchemy-utils 

In [ ]:
pip install python-dotenv

In [3]:
import pandas as pd 
import os
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy_utils import database_exists, create_database  
from dotenv import load_dotenv 
from pathlib import Path

Carga de datos de las tablas de Pedidos

In [ ]:
import os
import sys

# Forzar a Python y PostgreSQL a ignorar las codificaciones locales de Windows
os.environ['PGCLIENTENCODING'] = 'utf-8'
os.environ['LC_ALL'] = 'C'
if sys.platform == "win32":
    # Fuerza a la consola de Windows a usar UTF-8 de forma nativa
    os.system('chcp 65001 > nul')

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Cargar las variables de entorno desde el archivo .env
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def asegurar_base_datos():
    
    # 2. SEGUNDO BLINDAJE: Forzar el encoding en las opciones nativas de conexión

    try:
        conn = psycopg2.connect(
            host=db_host,
            port=db_port,
            user=db_user,
            password=db_password,
            database='postgres',  
            client_encoding='utf-8', # Forzamos UTF-8 aquí
            options="-c client_encoding=utf8" # Forzamos parámetros internos del servidor
        )
    except UnicodeDecodeError:
        # Si aun así falla por temas de caracteres, usamos 'latin1' que acepta cualquier byte de Windows
        conn = psycopg2.connect(
            host=db_host,
            port=db_port,
            user=db_user,
            password=db_password,
            database='postgres',  
            client_encoding='latin1'
        )
        
    conn.autocommit = True 
    cursor = conn.cursor()

    # Buscamos si la base de datos ya existe en el servidor
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f"La base de datos '{db_name}' no existe. Creando...")
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f"Base de datos '{db_name}' creada exitosamente.")
    else:
        print(f"Conexión verificada: La base de datos '{db_name}' ya existe y está lista.")

    cursor.close()
    conn.close()

def run_etl():
    # Asegurar que la base de datos exista
    asegurar_base_datos()

    # Crear la conexión a la base de datos definitiva
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)

    # Extraer los archivos de la carpeta origen
    carpeta_origen = r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Pedidos"
    if not os.path.exists(carpeta_origen):
        os.makedirs(carpeta_origen)
        print(f"La carpeta '{carpeta_origen}' no existe.")
        return
    
    # Buscar los archivos que terminan en .xlsx dentro de la carpeta origen
    archivos = [f for f in os.listdir(carpeta_origen) if f.endswith('.xlsx')]

    if archivos:
        df_list = [pd.read_excel(os.path.join(carpeta_origen, a)) for a in archivos]
        df_final = pd.concat(df_list, ignore_index=True)

        # Transformar los datos (limpieza)
        df_final.columns = [c.lower().replace(' ', '_').strip() for c in df_final.columns]
        
        # Cargar los datos en la base de datos
        print(f"Cargando datos en la base de datos en {db_name} ...")
        df_final.to_sql('pedidos', engine, if_exists='replace', index=False)
        print("Datos cargados exitosamente.")
    else:
        print(f"No se encontraron archivos XLSX en la carpeta '{carpeta_origen}'.")

if __name__ == "__main__":
    run_etl()

Conexión verificada: La base de datos 'ventasdb' ya existe y está lista.
Cargando datos en la base de datos en ventasdb ...
Datos cargados exitosamente.


Carga de tabla Clientes

In [5]:
# import os
# from pathlib import Path
# import pandas as pd
# import psycopg2
# from dotenv import load_dotenv
# from sqlalchemy import create_engine

# Cargar las variables de entorno
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def conexion_db():
    # Conexión inicial a la base genérica 'postgres' para verificar/crear la base de datos destino
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database='postgres',  
        client_encoding='utf-8', 
        options="-c client_encoding=utf8" 
    )

    conn.autocommit = True
    cursor = conn.cursor()

    # Verificar si la base de datos ya existe
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f'La base de datos "{db_name}" no existe. Creando....')
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f'Base de datos "{db_name}" creada exitosamente.')
    else:
        print(f'La base de datos "{db_name}" ya existe.')
        
    cursor.close()
    conn.close()

def run_etl():
    # 1. Asegurar que la base de datos exista
    conexion_db()

    # 2. Crear la conexión a la base de datos DEFINITIVA (usando db_name)
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)   

    # 3. Definir rutas usando pathlib de forma correcta
    carpeta_origen = Path(r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos")
    nombre_archivo = "Clientes.xlsx"
    ruta_tabla = carpeta_origen / nombre_archivo

    # 4. Proceso de Extracción y Transformación
    if ruta_tabla.exists():
        print(f"Archivo encontrado: {ruta_tabla}")
        
        # Cargar el archivo Excel
        df_clientes = pd.read_excel(ruta_tabla)
        
        # Transformación 1: Forzar minúsculas, cambiar espacios por guiones bajos y limpiar extremos
        df_clientes.columns = [c.lower().replace(' ', '_').strip() for c in df_clientes.columns]
        
        # Transformación 2: Renombrar columnas específicas de manera segura con Pandas
         # Nota: Mapeamos los nombres originales (ya en minúsculas por el paso anterior) a los nuevos nombres limpios.
        columnas_nuevas = {
            'Clientes.Cliente_ID': 'cliente_id',  
            'Clientes.Nombre': 'nombre', 
            'Clientes.Contacto': 'contacto',  
            'Clientes.Ciudad': 'ciudad', 
            'Clientes.Pais': 'pais',
            'Clientes.Division': 'division', 
            'Clientes.Direccion': 'direccion', 
            'Clientes.CodigoPostal': 'codigo_postal'
        }
        df_clientes = df_clientes.rename(columns=columnas_nuevas)

        # 5. Carga de datos en la base de datos
        print(f"Cargando datos en la tabla 'clientes' dentro de la base de datos '{db_name}' ...")
        # 'replace' sobreescribe la tabla si ya existe. Cambiar a 'append' si solo quieres sumar filas.
        df_clientes.to_sql('clientes', engine, if_exists='replace', index=False)
        print("¡Datos cargados exitosamente!")
        
    else:
        print(f"ERROR: No se encontró el archivo '{nombre_archivo}' en la carpeta '{carpeta_origen}'.")
        return

if __name__ == "__main__":
    run_etl()

La base de datos "ventasdb" ya existe.
Archivo encontrado: C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Clientes.xlsx
Cargando datos en la tabla 'clientes' dentro de la base de datos 'ventasdb' ...
¡Datos cargados exitosamente!


In [ ]:
# #para corregir 
# #Cargar el archivo de clientes
# load_dotenv()

# #Obtener las variables de entorno para la conexión
# db_user = os.getenv('db_user')
# db_password = os.getenv('db_password')
# db_host = os.getenv('db_host')
# db_port = os.getenv('db_port')
# db_name = os.getenv('db_name')

# def conexion_db():
#     # Canal de conexion 
#     conn = psycopg2.connect(
#         host=db_host,
#         port=db_port,
#         user=db_user,
#         password=db_password,
#         database='postgres',  
#         client_encoding='utf-8', 
#         options="-c client_encoding=utf8" 
#     )

#     conn.autocommit = True
#     cursor = conn.cursor()

#     cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
#     exists = cursor.fetchone()

#     if not exists:
#         print(f'La base de datos "{db_name}" no existe. Creando....')
#         cursor.execute(f"CREATE DATABASE {db_name}")
#         print(f'Base de datos "{db_name}" creada exitosamente.')
#     cursor.close()
#     conn.close()

# def run_etl():

#     conexion_db()

#     # Crear la conexion a la base de datos definitiva
#     url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/postgres'
#     engine = create_engine(url_conexion)   

#     # Extraer los archivos de la carpeta origen
#     carpeta_origen = r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos"
#     a_cliente = r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Clientes.xlsx"
#     ruta_tabla = carpeta_origen / a_cliente

#     if ruta_tabla.exists():
#         print(f"Archivo encontrado: {ruta_tabla}")
#     else:
#         print(f"Archivo no encontrado: {ruta_tabla}")
#         return
#     if a_cliente:
#         #Cargar el archivo
#         df_clientes = pd.read_excel(ruta_tabla)
#         # Transformar los datos (limpieza)
#         df_clientes.columns = [c.lower().replace(' ', '_').strip() for c in df_clientes.columns]
#         df_clientes.columns = [c.rename(columns={'Clientes.Cliente_ID': 'cliente_id',  'Clientes.Nombre': 'nombre', 'Clientes.Contacto': 'contacto',  'Clientes.Ciudad': 'ciudad', 'Clientes.Pais': 'pais','Clientes.Division': 'division', 'Clientes.Direccion': 'direccion', 'Clientes.CodigoPostal': 'codigopotal'}) for c in df_clientes.columns]

#         # Cargar los datos en la base de datos
#         print(f"Cargando datos en la base de datos en {db_name} ...")
#         df_clientes.to_sql('clientes', engine, if_exists='replace', index=False)
#         print("Datos cargados exitosamente.")
#     else:
#         print(f"No se encontró el archivo '{a_cliente}' en la carpeta '{carpeta_origen}'.")

# if __name__ == "__main__":
#     run_etl()


In [ ]:
# import os
# import pandas as pd
# import psycopg2
# from sqlalchemy import create_engine
# from dotenv import load_dotenv
# import urllib.parse

# # 1. Cargar variables de entorno
# load_dotenv()

# db_user = os.getenv('db_user')
# db_password = os.getenv('db_password')
# db_host = os.getenv('db_host')
# db_port = os.getenv('db_port')
# db_name = os.getenv('db_name')

# # Escapar la contraseña para la URL de SQLAlchemy
# password_escapada = urllib.parse.quote_plus(db_password if db_password else "")

# def asegurar_base_datos():
#     """Crea la base de datos si no existe conectándose primero a 'postgres'."""
#     try:
#         conn = psycopg2.connect(
#             host=db_host,
#             port=db_port,
#             user=db_user,
#             password=db_password,
#             database='postgres',
#             client_encoding='utf8'
#         )
#         conn.autocommit = True
#         cursor = conn.cursor()

#         cursor.execute("SELECT 1 FROM pg_database WHERE datname=%s", (db_name,))
#         if not cursor.fetchone():
#             print(f"Creando base de datos '{db_name}'...")
#             cursor.execute(f'CREATE DATABASE "{db_name}"')
#         else:
#             print(f"La base de datos '{db_name}' ya existe.")

#         cursor.close()
#         conn.close()
#     except Exception as e:
#         # Limpiador de errores para mensajes con tildes (byte 0xf3)
#         error_msg = str(e).encode('ascii', 'replace').decode('ascii')
#         print(f"Aviso en infraestructura: {error_msg}")

# def run_etl():
#     # Paso 1: Asegurar que la base de datos existe
#     asegurar_base_datos()

#     # Paso 2: Configurar motor de carga con el parámetro de encoding en la URL
#     # Añadir ?client_encoding=utf8 directamente a la cadena de conexión
#     url_conexion = f'postgresql://{db_user}:{password_escapada}@{db_host}:{db_port}/{db_name}?client_encoding=utf8'
#     engine = create_engine(url_conexion)

#     # Paso 3: Extraer (Extract)
#     base_path = r"C:\Users\casat\OneDrive\Documentos"
#     carpeta_origen = os.path.join(base_path, "Anibal personal", "ThePower Business School", "Power BI", "Caso Practico_Datos", "Pedidos")
    
#     if not os.path.exists(carpeta_origen):
#         print(f"❌ La carpeta NO EXISTE en: {carpeta_origen}")
#         return
    
#     archivos = [f for f in os.listdir(carpeta_origen) if f.lower().endswith('.xlsx')]

#     if archivos:
#         print(f"✅ Se encontraron {len(archivos)} archivos. Extrayendo datos...")
#         df_list = []
#         for a in archivos:
#             ruta_completa = os.path.join(carpeta_origen, a)
#             try:
#                 df = pd.read_excel(ruta_completa, engine='openpyxl')
#                 df_list.append(df)
#             except Exception as e:
#                 print(f"Error al leer el archivo {a}: {e}")
            
#         if not df_list:
#             return

#         df_final = pd.concat(df_list, ignore_index=True)
        
#         # Transformación básica: limpiar nombres de columnas
#         df_final.columns = [c.lower().replace(' ', '_') for c in df_final.columns]
        
#         # Paso 5: Cargar (Load)
#         try:
#             print(f"Intentando cargar {len(df_final)} filas en '{db_name}'...")
#             df_final.to_sql('pedidos', engine, if_exists='replace', index=False)
#             print("🚀 ¡Proceso ETL finalizado con éxito!")
            
#         except Exception as e:
#             # Capturamos el error real de SQLAlchemy y lo limpiamos de caracteres problemáticos
#             raw_error = str(e).encode('utf-8', 'replace').decode('latin-1', 'replace')
#             print("\n❌ --- ERROR DETECTADO EN LA CARGA ---")
#             # Buscamos palabras clave en el error para ayudarte
#             if "password authentication failed" in raw_error.lower():
#                 print("Motivo: La contraseña del usuario 'postgres' es incorrecta.")
#             elif "does not exist" in raw_error.lower():
#                 print("Motivo: La base de datos no existe o no se pudo crear.")
#             else:
#                 # Imprimir el error filtrando los bytes que rompen la consola
#                 print(f"Detalle técnico: {raw_error[:200]}...") 
#             print("--------------------------------------\n")
#     else:
#         print(f"⚠️ No hay archivos .xlsx en: {carpeta_origen}")

# if __name__ == "__main__":
#     run_etl()

In [ ]:
# import os
# import pandas as pd
# import psycopg2
# from sqlalchemy import create_engine
# from dotenv import load_dotenv
# import urllib.parse

# # 1. Cargar variables de entorno
# load_dotenv()

# db_user = os.getenv('db_user')
# db_password = os.getenv('db_password')
# db_host = os.getenv('db_host')
# db_port = os.getenv('db_port')
# db_name = os.getenv('db_name')

# # Escapar la contraseña para evitar errores con caracteres especiales
# password_escapada = urllib.parse.quote_plus(db_password if db_password else "")

# def limpiar_mensaje_error(e):
#     """Convierte errores con tildes (Latin-1) en texto legible para Python."""
#     try:
#         # Intentamos forzar la conversión de los bytes problemáticos
#         return str(e).encode('latin-1', errors='replace').decode('utf-8', errors='replace')
#     except:
#         # Si falla, simplemente eliminamos caracteres no ASCII
#         return str(e).encode('ascii', errors='ignore').decode('ascii')

# def asegurar_base_datos():
#     """Crea la base de datos si no existe."""
#     try:
#         conn = psycopg2.connect(
#             host=db_host,
#             port=db_port,
#             user=db_user,
#             password=db_password,
#             database=db_name 
#         )
#         conn.autocommit = True
#         cursor = conn.cursor()
#         cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
#         if not cursor.fetchone():
#             print(f"Creando base de datos '{db_name}'...")
#             cursor.execute(f'CREATE DATABASE "{db_name}"')
#         else:
#             print(f"La base de datos '{db_name}' ya existe.")
#         cursor.close()
#         conn.close()
#     except Exception as e:
#         print(f"Aviso en infraestructura: {limpiar_mensaje_error(e)}")

# def run_etl():
#     # Paso 1: Asegurar infraestructura
#     asegurar_base_datos()

#     # Paso 2: Configurar motor de carga
#     # Usamos la URL con el parámetro client_encoding directo
#     url_conexion = f'postgresql://{db_user}:{password_escapada}@{db_host}:{db_port}/{db_name}?client_encoding=utf8'
#     engine = create_engine(url_conexion)

#     # Paso 3: Extraer (Extract)
#     # IMPORTANTE: Revisa si 'Anibal' lleva tilde en tu Windows
#     base_path = r"C:\Users\casat\OneDrive\Documentos"
#     carpeta_origen = os.path.join(base_path, "Anibal personal", "ThePower Business School", "Power BI", "Caso Practico_Datos", "Pedidos")
    
#     if not os.path.exists(carpeta_origen):
#         print(f"❌ Carpeta no encontrada: {carpeta_origen}")
#         return
    
#     archivos = [f for f in os.listdir(carpeta_origen) if f.lower().endswith('.xlsx')]

#     if archivos:
#         print(f"📦 Procesando {len(archivos)} archivos Excel...")
#         try:
#             df_list = [pd.read_excel(os.path.join(carpeta_origen, a), engine='openpyxl') for a in archivos]
#             df_final = pd.concat(df_list, ignore_index=True)
#             df_final.columns = [c.lower().replace(' ', '_') for c in df_final.columns]
            
#             # Paso 5: Cargar (Load)
#             print(f"🚀 Intentando cargar {len(df_final)} filas en '{db_name}'...")
#             df_final.to_sql('pedidos', engine, if_exists='replace', index=False)
#             print("✨ ¡Proceso ETL finalizado con éxito!")
            
#         except Exception as e:
#             # Capturamos el error y lo limpiamos antes de imprimirlo
#             error_real = limpiar_mensaje_error(e)
#             print("\n❌ --- ERROR DE CONEXIÓN ---")
#             print(f"DETALLE: {error_real}")
#             print("----------------------------\n")
#     else:
#         print(f"⚠️ No se hallaron archivos .xlsx en {carpeta_origen}")

# if __name__ == "__main__":
#     run_etl()